# 04 · Revisão dos resultados: os números resistem?

**Fase do CRISP-DM:** 5. Avaliação (revisão crítica antes da apresentação)

Antes de levar os números ao hotel, fazemos o papel de "advogado do diabo" com duas perguntas
que um gerente atento faria:

1. **As linhas repetidas mudam o tamanho do problema?** O arquivo não tem número de reserva, e
   cerca de 1 em cada 4 linhas é idêntica a outra. Decidimos mantê-las (decisão D3), porque em
   geral são quartos de um mesmo grupo. Mas e se forem, na verdade, a mesma reserva repetida?
2. **O radar depende das reservas "não reembolsáveis"?** Elas cancelam 99% das vezes, um padrão
   estranho que ainda precisa ser investigado (notebook 02). Se esse dado estiver errado, o
   radar ainda funciona?

Nada aqui muda a base tratada nem o modelo dos notebooks 01 a 03: apenas medimos se as
conclusões continuam de pé. Os resultados vão para `reports/tables/04_*.csv`.

In [1]:
import sys
from pathlib import Path

RAIZ = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "requirements.txt").exists())
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

import etl
from etl import SEED, carregar_dados_tratados
from visual import salvar_tabela


def br(numero, casas=0):
    # Formata um número no padrão brasileiro: 119.390 · 37,1.
    texto = f"{numero:,.{casas}f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")


def pct(valor, casas=1):
    return f"{br(100 * valor, casas)}%"

## 1. As linhas repetidas mudam o tamanho do problema?

Partimos do arquivo original, com **todas** as colunas, e tiramos só as reservas impossíveis
(a mesma regra do ETL). Depois contamos as linhas que são **idênticas em tudo** a outra.

Calculamos os números da apresentação de dois jeitos:

- **Cada linha é uma reserva** (o que usamos no projeto);
- **Linhas idênticas contam uma vez só** (o cenário mais pessimista para a nossa decisão).

A verdade está entre os dois: parte das repetições é de grupos reais, parte pode ser
repetição de registro. Sem o número da reserva, não dá para separar.

In [2]:
bruto = etl.remover_impossiveis(etl.extrair())
bruto["receita_estimada"] = bruto["adr"] * (
    bruto["stays_in_weekend_nights"] + bruto["stays_in_week_nights"]
)
distintas = bruto.drop_duplicates(subset=[c for c in bruto.columns if c != "receita_estimada"])


def resumo(df):
    canceladas = df[df["is_canceled"] == 1]
    return {
        "reservas": len(df),
        "canceladas": len(canceladas),
        "taxa_cancelamento_pct": round(100 * df["is_canceled"].mean(), 2),
        "receita_cancelada_eur": round(canceladas["receita_estimada"].sum(), 2),
        "pct_receita_cancelada": round(
            100 * canceladas["receita_estimada"].sum() / df["receita_estimada"].sum(), 2
        ),
    }


repetidas = pd.DataFrame(
    [
        {"cenario": "Cada linha é uma reserva (usado no projeto)", **resumo(bruto)},
        {"cenario": "Linhas idênticas contam uma vez só", **resumo(distintas)},
    ]
)

# Conferência: o primeiro cenário tem de bater com o notebook 01.
assert repetidas.loc[0, "reservas"] == 119_208
assert repetidas.loc[0, "canceladas"] == 44_198

salvar_tabela(repetidas, "04_linhas_repetidas")
print(f"Linhas idênticas a outra: {br(len(bruto) - len(distintas))} "
      f"({pct((len(bruto) - len(distintas)) / len(bruto))} das linhas)")
repetidas

Linhas idênticas a outra: 31.980 (26,8% das linhas)


,cenario,reservas,canceladas,taxa_cancelamento_pct,receita_cancelada_eur,pct_receita_cancelada
0,Cada linha é uma reserva (usado no projeto),119208,44198,37.08,16721837.12,39.15
1,Linhas idênticas contam uma vez só,87228,24008,27.52,11478642.98,33.32


**Leitura:** mesmo no cenário mais pessimista, **mais de 1 em cada 4 reservas é cancelada** e
os cancelamentos somam milhões de euros. O tamanho exato muda, mas a conclusão de que **o
problema é grande** continua de pé. Na apresentação, mostramos os dois números.

### O valor das reservas canceladas não é todo "perdido"

A receita estimada é **diária × noites** das reservas canceladas. É o valor que *estava
reservado*, não o prejuízo: o quarto pode ter sido revendido, e nas reservas "não
reembolsáveis" o hotel pode ter ficado com o depósito. A base não diz nenhuma das duas coisas.

In [3]:
canceladas = bruto[bruto["is_canceled"] == 1]
nr_canceladas = canceladas[canceladas["deposit_type"] == "Non Refund"]
print(f"Receita estimada das canceladas: € {br(canceladas['receita_estimada'].sum() / 1e6, 1)} mi")
print(f"  das quais 'não reembolsáveis': € {br(nr_canceladas['receita_estimada'].sum() / 1e6, 1)} mi "
      f"({pct(nr_canceladas['receita_estimada'].sum() / canceladas['receita_estimada'].sum(), 0)})")

Receita estimada das canceladas: € 16,7 mi
  das quais 'não reembolsáveis': € 3,6 mi (22%)


## 2. As reservas "não reembolsáveis" de perto

In [4]:
nr = bruto[bruto["deposit_type"] == "Non Refund"]
nr_distintas = distintas[distintas["deposit_type"] == "Non Refund"]

nao_reembolsavel = pd.DataFrame(
    [
        {"indicador": "Linhas 'não reembolsáveis'", "valor": len(nr)},
        {"indicador": "Taxa de cancelamento (todas as linhas, %)",
         "valor": round(100 * nr["is_canceled"].mean(), 2)},
        {"indicador": "Linhas idênticas a outra (%)",
         "valor": round(100 * (1 - len(nr_distintas) / len(nr)), 2)},
        {"indicador": "Reservas distintas", "valor": len(nr_distintas)},
        {"indicador": "Taxa de cancelamento (reservas distintas, %)",
         "valor": round(100 * nr_distintas["is_canceled"].mean(), 2)},
        {"indicador": "Receita das canceladas que é 'não reembolsável' (%)",
         "valor": round(100 * nr_canceladas["receita_estimada"].sum()
                        / canceladas["receita_estimada"].sum(), 2)},
    ]
)
salvar_tabela(nao_reembolsavel, "04_nao_reembolsavel")
nao_reembolsavel

,indicador,valor
0,Linhas 'não reembolsáveis',14586.00
1,"Taxa de cancelamento (todas as linhas, %)",99.36
2,Linhas idênticas a outra (%),92.89
3,Reservas distintas,1037.00
4,"Taxa de cancelamento (reservas distintas, %)",94.70
5,Receita das canceladas que é 'não reembolsável...,21.52


**Leitura:** quase todas as linhas "não reembolsáveis" são repetidas. Contando cada uma uma vez
só, sobram cerca de mil reservas, e mesmo assim **quase todas cancelam**. O padrão continua
estranho e reforça a recomendação de investigar com a equipe do hotel, mas é menor do que o
número de linhas sugere.

## 3. O radar depende das reservas "não reembolsáveis"?

Reconstruímos **exatamente** o radar do notebook 03: a mesma floresta aleatória, o mesmo
limiar de alerta escolhido no treino e as mesmas faixas de risco. As conferências garantem que
os números batem com o notebook 03.

Depois fazemos dois testes de resistência nas reservas de 2017:

- **Teste A:** o mesmo radar, avaliado **só nas reservas que não são "não reembolsáveis"**.
- **Teste B:** um radar treinado **sem olhar o tipo de depósito**, avaliado nessas mesmas
  reservas. É o que o hotel teria se descobrisse que esse campo está errado.

A comparação é sempre com a **regra do gerente** nas mesmas reservas. Nelas, a regra vira
"cancela se a reserva foi feita com mais de 90 dias de antecedência".

In [5]:
base = carregar_dados_tratados()
treino = base[base["conjunto"] == "treino"].reset_index(drop=True)
teste = base[base["conjunto"] == "teste"].reset_index(drop=True)

NUMERICAS = [
    "antecedencia_dias", "mes_chegada", "semana_chegada", "noites_fim_semana", "noites_semana",
    "adultos", "criancas", "bebes", "hospede_recorrente", "cancelamentos_anteriores",
    "reservas_anteriores_nao_canceladas", "dias_lista_espera", "diaria_media",
    "vagas_estacionamento", "pedidos_especiais", "tem_agente", "tem_empresa",
]
CATEGORICAS = [
    "hotel", "regime_refeicao", "segmento_mercado", "canal_distribuicao",
    "tipo_quarto_reservado", "tipo_deposito", "tipo_cliente",
]
FAIXAS = ["Baixo", "Médio", "Alto"]


def radar(categoricas):
    # Mesma configuração do modelo escolhido no notebook 03.
    return Pipeline(
        [
            ("preparo", ColumnTransformer(
                [("numeros", "passthrough", NUMERICAS),
                 ("categorias", OneHotEncoder(handle_unknown="ignore"), categoricas)]
            )),
            ("modelo", RandomForestClassifier(
                n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED
            )),
        ]
    )


def treinar(categoricas):
    # Treina em 2015–2016, escolhe o limiar no treino (como no notebook 03) e pontua 2017.
    atributos = NUMERICAS + categoricas
    modelo = radar(categoricas)
    validacao = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    prob_treino = cross_val_predict(
        modelo, treino[atributos], treino["cancelada"], cv=validacao, method="predict_proba"
    )[:, 1]
    precisoes, coberturas, limiares = precision_recall_curve(treino["cancelada"], prob_treino)
    f1s = 2 * precisoes * coberturas / np.clip(precisoes + coberturas, 1e-12, None)
    limiar = float(limiares[int(np.nanargmax(f1s[:-1]))])
    modelo.fit(treino[atributos], treino["cancelada"])
    return modelo.predict_proba(teste[atributos])[:, 1], limiar


def faixa_de_risco(probabilidade):
    return pd.cut(np.round(probabilidade, 3), bins=[-0.001, 0.30, 0.60, 1.001],
                  right=False, labels=FAIXAS)


def regra_do_gerente(df):
    return ((df["antecedencia_dias"] > 90) | (df["tipo_deposito"] == "Não reembolsável")).astype(int)


def avaliar(cenario, mascara, prob, limiar):
    y = teste.loc[mascara, "cancelada"]
    aviso = (prob[mascara] >= limiar).astype(int)
    regra = regra_do_gerente(teste[mascara])
    faixa = faixa_de_risco(prob[mascara])
    alto = y[np.asarray(faixa == "Alto")]
    return {
        "cenario": cenario,
        "reservas_2017": int(mascara.sum()),
        "f1_radar": round(f1_score(y, aviso), 4),
        "f1_regra_gerente": round(f1_score(y, regra), 4),
        "precisao_radar": round(precision_score(y, aviso), 4),
        "precisao_regra_gerente": round(precision_score(y, regra), 4),
        "cobertura_radar": round(recall_score(y, aviso), 4),
        "cobertura_regra_gerente": round(recall_score(y, regra), 4),
        "reservas_faixa_alta": len(alto),
        "taxa_cancelamento_faixa_alta": round(alto.mean(), 4),
    }


prob_original, limiar_original = treinar(CATEGORICAS)
prob_sem_deposito, limiar_sem_deposito = treinar([c for c in CATEGORICAS if c != "tipo_deposito"])
print(f"Limiar do radar original: {pct(limiar_original)} · "
      f"do radar sem depósito: {pct(limiar_sem_deposito)}")

Limiar do radar original: 37,1% · do radar sem depósito: 38,2%


In [6]:
todas = np.ones(len(teste), dtype=bool)
sem_nao_reembolsavel = (teste["tipo_deposito"] != "Não reembolsável").to_numpy()

robustez = pd.DataFrame(
    [
        avaliar("Radar original, todas as reservas de 2017", todas,
                prob_original, limiar_original),
        avaliar("A. Radar original, sem as 'não reembolsáveis'", sem_nao_reembolsavel,
                prob_original, limiar_original),
        avaliar("B. Radar treinado sem o depósito, sem as 'não reembolsáveis'",
                sem_nao_reembolsavel, prob_sem_deposito, limiar_sem_deposito),
    ]
)

# Conferências: a primeira linha reproduz o notebook 03 (03_criterio_de_sucesso e 03_faixas_de_risco).
original = robustez.iloc[0]
assert round(original["f1_radar"], 2) == 0.67 and round(original["f1_regra_gerente"], 2) == 0.58
assert original["reservas_faixa_alta"] == 9_372
assert round(original["taxa_cancelamento_faixa_alta"], 3) == 0.812

salvar_tabela(robustez, "04_robustez_do_radar")
robustez.set_index("cenario").T

cenario,"Radar original, todas as reservas de 2017","A. Radar original, sem as 'não reembolsáveis'","B. Radar treinado sem o depósito, sem as 'não reembolsáveis'"
reservas_2017,40619.0000,36493.0000,36493.0000
f1_radar,0.6729,0.5425,0.5393
f1_regra_gerente,0.5840,0.4609,0.4609
precisao_radar,0.7396,0.6212,0.6104
precisao_regra_gerente,0.5181,0.3958,0.3958
cobertura_radar,0.6172,0.4815,0.4830
cobertura_regra_gerente,0.6689,0.5516,0.5516
reservas_faixa_alta,9372.0000,5257.0000,5365.0000
taxa_cancelamento_faixa_alta,0.8121,0.6650,0.6675


In [7]:
for _, linha in robustez.iterrows():
    print(linha["cenario"])
    print(f"  Nota F1: radar {br(linha['f1_radar'], 2)} × regra do gerente "
          f"{br(linha['f1_regra_gerente'], 2)}")
    print(f"  Faixa alta: {pct(linha['taxa_cancelamento_faixa_alta'], 0)} cancelaram "
          f"({br(linha['reservas_faixa_alta'])} reservas)")

Radar original, todas as reservas de 2017
  Nota F1: radar 0,67 × regra do gerente 0,58
  Faixa alta: 81% cancelaram (9.372 reservas)
A. Radar original, sem as 'não reembolsáveis'
  Nota F1: radar 0,54 × regra do gerente 0,46
  Faixa alta: 66% cancelaram (5.257 reservas)
B. Radar treinado sem o depósito, sem as 'não reembolsáveis'
  Nota F1: radar 0,54 × regra do gerente 0,46
  Faixa alta: 67% cancelaram (5.365 reservas)


**Leitura:**

- **O critério 1 resiste.** Sem as reservas "não reembolsáveis", o radar **continua melhor que a
  regra do gerente**, com praticamente a mesma vantagem (cerca de 0,08 ponto de nota F1). Isso vale mesmo para um radar que nunca viu o
  tipo de depósito (teste B). Ou seja, o radar aprendeu padrões reais: antecedência, pedidos
  especiais, canal, histórico.
- **O critério 2 fica perto, mas abaixo da meta.** Sem essas reservas, a faixa alta tem **cerca
  de 2 em cada 3** reservas canceladas, um pouco abaixo da meta de 7 em 10. Parte do "81%" vem,
  portanto, das reservas "não reembolsáveis".
- **O que isso muda:** o radar supera a regra em qualquer cenário, mas a taxa da faixa alta
  depende do que se descobrir sobre as reservas "não reembolsáveis": "8 em cada 10" ou
  "2 em cada 3".